# Version 11: Bi-GRU + CRF for Protein Secondary Structure

**Target: F1 Harmonic ≥ 0.60**

## Key Improvements over V10:
- **CRF decoder for Q8** - Learns transition rules (helices ≥4 residues, etc.)
- **Physicochemical embeddings** - Hydrophobicity, charge, mass, pI
- **Differential learning rates** - Backbone 2e-3, CRF 5e-4
- **No label smoothing** - CRF handles regularization

In [ ]:
!pip install -q pytorch-crf trackio

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.amp import autocast  # Updated API
from torch.cuda.amp import GradScaler
from torchcrf import CRF
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True

class Config:
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Architecture
    embed_dim = 128
    physico_dim = 4  # Hydrophobicity, charge, mass, pI
    cnn_channels = 64
    hidden_dim = 384
    num_layers = 2
    dropout = 0.5  # Increased for regularization
    
    # Training
    batch_size = 32
    grad_accum = 2  # Effective batch = 64
    lr_backbone = 2e-3  # Higher for backbone
    lr_crf = 5e-4  # Lower for CRF
    weight_decay = 1e-4
    epochs = 60
    grad_clip = 5.0  # Higher for CRF
    use_amp = True
    
    val_split = 0.1
    train_path = 'train.csv'
    test_path = 'test.csv'
    model_save_path = 'best_model_bigru_v11_crf.pth'
    submission_path = 'submission_bigru_v11_crf.csv'
    
    trackio_project = '25-t3-nppe2'
    trackio_space_id = 'manyakaistha/dlgenai-nppe'
    trackio_group = 'bi_gru_v11_crf'

cfg = Config()
print(f'Device: {cfg.device}')

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

In [ ]:
os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'hf_placeholder_token_for_reproducibility'
os.environ['TRACKIO_SPACE_ID'] = cfg.trackio_space_id

def maybe_init_trackio():
    try:
        import trackio
        if cfg.trackio_space_id:
            trackio.init(project=cfg.trackio_project, space_id=cfg.trackio_space_id, group=cfg.trackio_group)
            print('✓ TrackIO initialized')
            return trackio
    except Exception as e:
        print(f'TrackIO: {e}')
    return None

trk = maybe_init_trackio()

In [ ]:
# Amino acid physicochemical properties: [Hydrophobicity, Charge, Mass(scaled), pI(scaled)]
# Hydrophobicity: Kyte-Doolittle scale (-4.5 to 4.5, scaled to ~-1 to 1)
# Charge: -1, 0, +1
# Mass: scaled by /200
# pI: scaled by /14

AA_PROPS = {
    '<PAD>': [0, 0, 0, 0], '<UNK>': [0, 0, 0, 0], '*': [0, 0, 0, 0],
    'A': [0.4, 0, 0.45, 0.43], 'R': [-1.0, 1, 0.87, 0.77],
    'N': [-0.78, 0, 0.66, 0.39], 'D': [-0.78, -1, 0.67, 0.20],
    'C': [0.56, 0, 0.61, 0.36], 'E': [-0.78, -1, 0.74, 0.23],
    'Q': [-0.78, 0, 0.73, 0.40], 'G': [-0.09, 0, 0.38, 0.43],
    'H': [-0.71, 0.5, 0.78, 0.54], 'I': [1.0, 0, 0.66, 0.43],
    'L': [0.84, 0, 0.66, 0.43], 'K': [-0.87, 1, 0.73, 0.70],
    'M': [0.42, 0, 0.75, 0.41], 'F': [0.62, 0, 0.83, 0.39],
    'P': [-0.36, 0, 0.58, 0.45], 'S': [-0.18, 0, 0.53, 0.41],
    'T': [-0.16, 0, 0.60, 0.40], 'W': [-0.20, 0, 1.02, 0.42],
    'Y': [-0.29, 0, 0.91, 0.40], 'V': [0.93, 0, 0.59, 0.43]
}

# Vocabularies
PAD_TOKEN = 0
AA_CHARS = ['<PAD>', '<UNK>', '*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
AA_TO_ID = {c: i for i, c in enumerate(AA_CHARS)}
ID_TO_AA = {i: c for c, i in AA_TO_ID.items()}

Q8_CHARS = ['<PAD>'] + sorted(['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T'])
Q8_TO_ID = {c: i for i, c in enumerate(Q8_CHARS)}
ID_TO_Q8 = {i: c for c, i in Q8_TO_ID.items()}

Q3_CHARS = ['<PAD>'] + sorted(['C', 'E', 'H'])
Q3_TO_ID = {c: i for i, c in enumerate(Q3_CHARS)}
ID_TO_Q3 = {i: c for c, i in Q3_TO_ID.items()}

# Create property tensor
prop_tensor = torch.zeros(len(AA_CHARS), cfg.physico_dim)
for char, idx in AA_TO_ID.items():
    if char in AA_PROPS:
        prop_tensor[idx] = torch.tensor(AA_PROPS[char])

print(f'Vocab: AA={len(AA_TO_ID)}, Q8={len(Q8_TO_ID)}, Q3={len(Q3_TO_ID)}')
print(f'Property tensor shape: {prop_tensor.shape}')

In [ ]:
class ProteinDataset(Dataset):
    def __init__(self, df, mode='train'):
        self.df = df
        self.mode = mode
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = [AA_TO_ID.get(c, AA_TO_ID['<UNK>']) for c in row['seq']]
        item = {'seq': torch.tensor(seq, dtype=torch.long), 'id': row['id'], 'len': len(seq)}
        if self.mode == 'train':
            item['q8'] = torch.tensor([Q8_TO_ID[c] for c in row['sst8']], dtype=torch.long)
            item['q3'] = torch.tensor([Q3_TO_ID[c] for c in row['sst3']], dtype=torch.long)
        return item

def collate_fn(batch):
    batch.sort(key=lambda x: x['len'], reverse=True)  # Sort for efficiency
    seqs = pad_sequence([x['seq'] for x in batch], batch_first=True, padding_value=PAD_TOKEN)
    lengths = torch.tensor([x['len'] for x in batch])
    out = {'seq': seqs, 'len': lengths, 'ids': [x['id'] for x in batch]}
    if 'q8' in batch[0]:
        out['q8'] = pad_sequence([x['q8'] for x in batch], batch_first=True, padding_value=PAD_TOKEN)
        out['q3'] = pad_sequence([x['q3'] for x in batch], batch_first=True, padding_value=PAD_TOKEN)
    return out

df_full = pd.read_csv(cfg.train_path)
train_df, val_df = train_test_split(df_full, test_size=cfg.val_split, random_state=cfg.seed)

train_loader = DataLoader(ProteinDataset(train_df), batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=2)
val_loader = DataLoader(ProteinDataset(val_df), batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=2)

print(f'Train: {len(train_df)}, Val: {len(val_df)}')

In [ ]:
class BiGRU_CRF(nn.Module):
    """Bi-GRU + CNN with CRF for Q8 and Softmax for Q3"""
    
    def __init__(self):
        super().__init__()
        
        # Embeddings
        self.embedding = nn.Embedding(len(AA_TO_ID), cfg.embed_dim, padding_idx=PAD_TOKEN)
        self.prop_embed = nn.Embedding.from_pretrained(prop_tensor, freeze=False)  # Trainable
        self.prop_proj = nn.Linear(cfg.physico_dim, 16)  # Project to higher dim
        
        # CNN for local context
        input_dim = cfg.embed_dim + 16
        self.conv1 = nn.Conv1d(input_dim, cfg.cnn_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(cfg.cnn_channels, cfg.cnn_channels, kernel_size=5, padding=2)
        self.cnn_norm = nn.LayerNorm(cfg.cnn_channels)
        self.dropout1 = nn.Dropout(cfg.dropout)
        
        # Bi-GRU
        self.gru = nn.GRU(
            input_size=cfg.cnn_channels,
            hidden_size=cfg.hidden_dim,
            num_layers=cfg.num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg.dropout if cfg.num_layers > 1 else 0
        )
        self.gru_norm = nn.LayerNorm(cfg.hidden_dim * 2)
        self.dropout2 = nn.Dropout(cfg.dropout)
        
        # Heads
        self.q8_fc = nn.Linear(cfg.hidden_dim * 2, len(Q8_TO_ID))
        self.q3_fc = nn.Linear(cfg.hidden_dim * 2, len(Q3_TO_ID))
        
        # CRF for Q8 only
        self.crf_q8 = CRF(len(Q8_TO_ID), batch_first=True)
    
    def _encode(self, x):
        # Embeddings
        embed = self.embedding(x)  # [B, L, embed_dim]
        props = self.prop_proj(self.prop_embed(x))  # [B, L, 16]
        x_in = torch.cat([embed, props], dim=2)  # [B, L, embed_dim+16]
        
        # CNN
        x_in = x_in.permute(0, 2, 1)  # [B, D, L]
        c1 = torch.relu(self.conv1(x_in))
        c2 = torch.relu(self.conv2(c1))
        x_conv = c2.permute(0, 2, 1)  # [B, L, cnn_channels]
        x_conv = self.dropout1(self.cnn_norm(x_conv))
        
        # GRU
        gru_out, _ = self.gru(x_conv)
        gru_out = self.dropout2(self.gru_norm(gru_out))
        
        return gru_out
    
    def forward(self, x, mask, q8_tags=None, q3_tags=None):
        gru_out = self._encode(x)
        
        # Emissions
        q8_emissions = self.q8_fc(gru_out)
        q3_emissions = self.q3_fc(gru_out)
        
        if q8_tags is not None:
            # Training mode: Return losses
            # CRF returns negative log-likelihood SUMMED over sequence
            # We need to normalize by total tokens to get per-token loss
            crf_nll = -self.crf_q8(q8_emissions, q8_tags, mask=mask, reduction='sum')
            total_tokens = mask.sum()  # Count non-padded tokens
            loss_q8 = crf_nll / total_tokens  # Per-token CRF loss (~2-3 expected)
            
            # CrossEntropy for Q3 (already per-token)
            loss_q3 = nn.functional.cross_entropy(
                q3_emissions.view(-1, len(Q3_TO_ID)),
                q3_tags.view(-1),
                ignore_index=PAD_TOKEN
            )
            return loss_q8, loss_q3
        else:
            # Inference mode: Decode
            pred_q8 = self.crf_q8.decode(q8_emissions, mask=mask)
            pred_q3 = q3_emissions.argmax(dim=-1)
            return pred_q8, pred_q3


model = BiGRU_CRF().to(cfg.device)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Differential learning rates
crf_params = list(model.crf_q8.parameters())
backbone_params = [p for n, p in model.named_parameters() if 'crf' not in n]

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': cfg.lr_backbone},
    {'params': crf_params, 'lr': cfg.lr_crf}
], weight_decay=cfg.weight_decay)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
scaler = GradScaler(enabled=cfg.use_amp)

class Checkpoint:
    def __init__(self, path):
        self.path = path
        self.best = -np.inf
    
    def __call__(self, score, model, epoch):
        if score > self.best:
            self.best = score
            torch.save({'epoch': epoch, 'model': model.state_dict(), 'score': score}, self.path)
            print(f'✓ Saved best model (epoch {epoch+1}, score={score:.4f})')
            return True
        return False

ckpt = Checkpoint(cfg.model_save_path)
print(f'Backbone LR: {cfg.lr_backbone}, CRF LR: {cfg.lr_crf}')

In [ ]:
def harmonic_mean(a, b):
    return 2 * (a * b) / (a + b + 1e-8)

def train_epoch(model, loader, optimizer, scheduler, scaler, device, epoch):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} Train')
    for step, batch in enumerate(pbar):
        seq = batch['seq'].to(device, non_blocking=True)
        mask = (seq != PAD_TOKEN)  # Boolean mask for CRF
        q8 = batch['q8'].to(device, non_blocking=True)
        q3 = batch['q3'].to(device, non_blocking=True)
        
        with autocast(device_type='cuda', enabled=cfg.use_amp):
            loss_q8, loss_q3 = model(seq, mask, q8, q3)
            loss = (loss_q8 + loss_q3) / cfg.grad_accum
        
        scaler.scale(loss).backward()
        
        if (step + 1) % cfg.grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step(epoch + step / len(loader))
        
        total_loss += (loss_q8.item() + loss_q3.item())
        pbar.set_postfix({'loss': total_loss / (step + 1)})
    
    torch.cuda.empty_cache()
    return total_loss / len(loader)


def validate(model, loader, device):
    model.eval()
    all_q8_p, all_q8_t = [], []
    all_q3_p, all_q3_t = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            seq = batch['seq'].to(device)
            mask = (seq != PAD_TOKEN)
            lengths = batch['len']
            
            # CRF decode returns list of lists
            with autocast(device_type='cuda', enabled=cfg.use_amp):
                pred_q8_list, pred_q3 = model(seq, mask)
            
            # Collect predictions
            for j, (p8_list, length) in enumerate(zip(pred_q8_list, lengths)):
                L = length.item()
                all_q8_p.extend(p8_list[:L])
                all_q8_t.extend(batch['q8'][j, :L].cpu().numpy())
                all_q3_p.extend(pred_q3[j, :L].cpu().numpy())
                all_q3_t.extend(batch['q3'][j, :L].cpu().numpy())
    
    torch.cuda.empty_cache()
    f1_q8 = f1_score(all_q8_t, all_q8_p, average='macro', zero_division=0)
    f1_q3 = f1_score(all_q3_t, all_q3_p, average='macro', zero_division=0)
    return f1_q8, f1_q3, harmonic_mean(f1_q8, f1_q3)

In [ ]:
history = {'loss': [], 'f1_q8': [], 'f1_q3': [], 'f1_harm': []}
best_score = 0

print('Starting training with CRF decoder...')
print(f'Target: F1 Harmonic ≥ 0.60\n')

for epoch in range(cfg.epochs):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, cfg.device, epoch)
    f1_q8, f1_q3, f1_harm = validate(model, val_loader, cfg.device)
    
    history['loss'].append(train_loss)
    history['f1_q8'].append(f1_q8)
    history['f1_q3'].append(f1_q3)
    history['f1_harm'].append(f1_harm)
    
    print(f'\nEpoch {epoch+1}/{cfg.epochs}:')
    print(f'  Loss: {train_loss:.4f} | Q8: {f1_q8:.4f} | Q3: {f1_q3:.4f} | Harm: {f1_harm:.4f}')
    
    ckpt(f1_harm, model, epoch)
    if f1_harm > best_score:
        best_score = f1_harm
    
    if trk:
        trk.log({'epoch': epoch+1, 'loss': train_loss, 'f1_q8': f1_q8, 'f1_q3': f1_q3, 'f1_harm': f1_harm})
    
    print('-' * 50)

print(f'\n✓ Training complete! Best F1 Harmonic: {best_score:.4f}')
print(f'Target achieved: {"✓ YES" if best_score >= 0.60 else "✗ NO"}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['loss']); ax[0].set_title('Loss'); ax[0].grid(True)
ax[1].plot(history['f1_harm'], label='Harmonic'); ax[1].plot(history['f1_q8'], '--', label='Q8')
ax[1].plot(history['f1_q3'], '--', label='Q3'); ax[1].axhline(0.60, color='r', linestyle=':')
ax[1].set_title('F1 Scores'); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.savefig('training_v11.png', dpi=150); plt.show()

In [ ]:
print('Loading best model...')
loaded = torch.load(cfg.model_save_path, map_location=cfg.device)
model.load_state_dict(loaded['model'])
print(f'✓ Loaded epoch {loaded["epoch"]+1}, score: {loaded["score"]:.4f}')

df_test = pd.read_csv(cfg.test_path)
test_loader = DataLoader(ProteinDataset(df_test, mode='test'), batch_size=cfg.batch_size*2, collate_fn=collate_fn)

model.eval()
results = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Inference'):
        seq = batch['seq'].to(cfg.device)
        mask = (seq != PAD_TOKEN)
        lengths = batch['len']
        
        with autocast(device_type='cuda', enabled=cfg.use_amp):
            pred_q8_list, pred_q3 = model(seq, mask)
        
        for j, (p8_list, length) in enumerate(zip(pred_q8_list, lengths)):
            L = length.item()
            str_q8 = ''.join([ID_TO_Q8[x] for x in p8_list[:L]])
            str_q3 = ''.join([ID_TO_Q3[x] for x in pred_q3[j, :L].cpu().numpy()])
            results.append({'id': batch['ids'][j], 'sst8': str_q8, 'sst3': str_q3})

df_sub = pd.DataFrame(results)
df_sub.to_csv(cfg.submission_path, index=False)
print(f'\n✓ Saved: {cfg.submission_path}')
print(df_sub.head())

In [ ]:
print('Validating...')
assert len(df_sub) == len(df_test)
for _, row in df_sub.iterrows():
    seq_len = len(df_test.loc[df_test['id'] == row['id'], 'seq'].values[0])
    assert len(row['sst8']) == seq_len and len(row['sst3']) == seq_len
print('✓ Valid!')